In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# List the contents of the root directory of your Google Drive
print(os.listdir('/content/drive/MyDrive/KY_9/KY_9_Final_Project/Data/processed/data_final_train_v2/'))

['data_final_sort_v2.jsonl', 'data_final_v2.txt', 'data_final_sort_v2_part_01.jsonl', 'data_final_sort_v2_part_02.jsonl', 'clean_data.py', 'data_final_sort_v2.txt', 'data_final_sort_v2_output.txt', 'data_final_sort_v2_fisrt_se.jsonl', 'corpus_chucks_v2.py']


In [4]:
file_path = '/content/drive/MyDrive/KY_9/KY_9_Final_Project/Data/processed/data_final_train_v2/data_final_sort_v2_fisrt_se.jsonl'

In [5]:
import json
import pickle
from sentence_transformers import SentenceTransformer

In [6]:
passages = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            obj = json.loads(line)
            passages.append(obj["passage"])
        except json.JSONDecodeError:
            print(" Lỗi JSON ở dòng:", len(passages))
            continue

print(" Đọc được:", len(passages))

 Đọc được: 15071


**bkai_foundation_models_fine_tuning**

In [ ]:
model_path_bkai = '/content/drive/MyDrive/KY_9/KY_9_Final_Project/Data/train_model/bkai_foundation_models_fine_tuning_v2'
model_bkai = SentenceTransformer(model_path_bkai)
#
output_path_bkai = '/content/drive/MyDrive/KY_9/KY_9_Final_Project/Data/embeddings/model_fine_tuning/embeding_by_model_fine_bkai_v2_copy.pkl'
#  Tạo thư mục nếu chưa tồn tại
os.makedirs(os.path.dirname(output_path_bkai), exist_ok=True)

In [ ]:
# Tạo embedding (batch encode)
embeddings = model_bkai .encode(passages, show_progress_bar=True)

Batches:   0%|          | 0/471 [00:00<?, ?it/s]

In [ ]:
from IPython.display import FileLink

os.makedirs(os.path.dirname(model_path_bkai), exist_ok=True)
with open(output_path_bkai, "wb") as f:
    pickle.dump(embeddings, f)

print(" Đã lưu:", output_path_bkai)
FileLink(output_path_bkai)


 Đã lưu: /content/drive/MyDrive/KY_9/KY_9_Final_Project/Data/embeddings/model_fine_tuning/embeding_by_model_fine_bkai_v2_copy.pkl


/content/drive/MyDrive/KY_9/KY_9_Final_Project/Data/embeddings/model_fine_tuning/embeding_by_model_fine_bkai_v2_copy.pkl

**intfloat_multilingual_e5_base**

In [ ]:
FT_DIR_E5 = "/content/drive/MyDrive/KY_9/KY_9_Final_Project/Data/train_model/intfloat-multilingual-e5-base_fine_tuning_v2"
model_e5 = SentenceTransformer(FT_DIR_E5)
output_path_e5 = '/content/drive/MyDrive/KY_9/KY_9_Final_ProjectData/embeddings/model_fine_tuning/embeding_by_model_fine_e5_v2.pkl'
#  Tạo thư mục nếu chưa tồn tại
os.makedirs(os.path.dirname(output_path_e5), exist_ok=True)

In [ ]:
# import shutil
# import torch
# import os
# from sentence_transformers import SentenceTransformer
# from safetensors.torch import load_file
# from collections import OrderedDict

# # 1. Đường dẫn
# FT_DIR_E5 = "/content/drive/MyDrive/KY_9/KY_9_Final_Project/Data/train_model/intfloat_multilingual_e5_base_fine_tuning_v2"
# SAFE_PATH = os.path.join(FT_DIR_E5, "model.safetensors")

# print(" Đang chẩn đoán lệch Key...")

# # 2. Tải model gốc (Cái khung)
# base_model = SentenceTransformer('intfloat/multilingual-e5-base')
# base_keys = list(base_model.state_dict().keys())
# print(f" Model mong đợi keys bắt đầu bằng: '{base_keys[0]}'")

# # 3. Tải file trọng số của bạn
# if os.path.exists(SAFE_PATH):
#     file_state_dict = load_file(SAFE_PATH)
#     file_keys = list(file_state_dict.keys())
#     print(f" File của bạn có keys bắt đầu bằng: '{file_keys[0]}'")

#     # 4. SO SÁNH VÀ SỬA
#     new_state_dict = OrderedDict()

#     # Trường hợp 1: File thiếu tiền tố "0.auto_model." (Rất hay gặp)
#     if not file_keys[0].startswith("0.") and base_keys[0].startswith("0.auto_model"):
#         print(" PHÁT HIỆN LỆCH: File thiếu tiền tố '0.auto_model.'")
#         print(" Đang tự động thêm tiền tố...")
#         for k, v in file_state_dict.items():
#             # Thường E5 save weights của transformer core, nên cần map vào 0.auto_model
#             new_key = f"0.auto_model.{k}"
#             new_state_dict[new_key] = v

#     # Trường hợp 2: File dùng prefix "model." (Do HuggingFace Trainer lưu)
#     elif file_keys[0].startswith("model."):
#         print(" PHÁT HIỆN LỆCH: File dùng prefix 'model.' thay vì '0.auto_model.'")
#         print(" Đang tự động đổi tên...")
#         for k, v in file_state_dict.items():
#             new_key = k.replace("model.", "0.auto_model.")
#             new_state_dict[new_key] = v

#     else:
#         print("ℹ Cấu trúc có vẻ khớp hoặc thuộc trường hợp lạ. Sẽ thử nạp trực tiếp.")
#         new_state_dict = file_state_dict

#     # 5. NẠP LẠI VỚI STRICT=TRUE (Để bắt buộc phải khớp)
#     print("\n Đang nạp trọng số đã sửa vào model (Strict Mode)...")
#     try:
#         # Load với strict=False nhưng hy vọng keys đã khớp phần lớn
#         missing, unexpected = base_model.load_state_dict(new_state_dict, strict=False)

#         print(" Nạp thành công!")
#         if len(missing) > 0:
#             print(f"   - Missing keys (chấp nhận được nếu là pooler): {len(missing)}")
#             # print(missing[:3]) # Uncomment để soi kỹ
#         if len(unexpected) > 0:
#             print(f"   - Unexpected keys (Keys thừa): {len(unexpected)}")

#         # Gán lại vào biến chính để đánh giá
#         model_e5 = base_model

#     except Exception as e:
#         print(f" VẪN LỖI KHI NẠP: {e}")

# else:
#     print(" Không tìm thấy file model.safetensors")
# output_path_e5 = '/content/drive/MyDrive/KY_9/KY_9_Final_ProjectData/embeddings/model_fine_tuning/embeding_by_model_fine_e5_v2.pkl'
# #  Tạo thư mục nếu chưa tồn tại
# os.makedirs(os.path.dirname(output_path_e5), exist_ok=True)


In [ ]:
# Tạo embedding (batch encode)
embeddings = model_e5.encode(passages, show_progress_bar=True)

In [ ]:
# Tạo embedding (batch encode)
embeddings = model_e5 .encode(passages, show_progress_bar=True)

In [ ]:
from IPython.display import FileLink

os.makedirs(os.path.dirname(output_path_e5), exist_ok=True)
with open(output_path_e5, "wb") as f:
    pickle.dump(embeddings, f)

print(" Đã lưu:", output_path_e5)
FileLink(output_path_e5)


**mpnet-base-v2_fine_tuning**

In [ ]:
model_path_mpnel_base_v2 = '/content/drive/MyDrive/KY_9/KY_9_Final_Project/Data/train_model/mpnet-base-v2_fine_tuning_v2'
model_mpnel_base_v2 = SentenceTransformer(model_path_mpnel_base_v2)
#
output_path_mpnel_base_v2 = '/content/drive/MyDrive/KY_9/KY_9_Final_Project/Data/embeddings/model_fine_tuning/embeding_by_model_fine_mpnet_base_v2_v2.pkl'
#  Tạo thư mục nếu chưa tồn tại
os.makedirs(os.path.dirname(output_path_mpnel_base_v2), exist_ok=True)

In [ ]:
# Tạo embedding (batch encode)
embeddings = model_mpnel_base_v2.encode(passages, show_progress_bar=True)

Batches:   0%|          | 0/471 [00:00<?, ?it/s]

In [ ]:
from IPython.display import FileLink
os.makedirs(os.path.dirname(output_path_mpnel_base_v2), exist_ok=True)
with open(output_path_mpnel_base_v2, "wb") as f:
    pickle.dump(embeddings, f)

print(" Đã lưu:", output_path_mpnel_base_v2)
FileLink(output_path_mpnel_base_v2)


 Đã lưu: /content/drive/MyDrive/KY_9/KY_9_Final_Project/Data/embeddings/model_fine_tuning/embeding_by_model_fine_mpnet_base_v2_v2.pkl


/content/drive/MyDrive/KY_9/KY_9_Final_Project/Data/embeddings/model_fine_tuning/embeding_by_model_fine_mpnet_base_v2_v2.pkl